# Ejercicio AirBnB: Extracción de entidades

En este cuaderno vamos a trabajar con un dataset de AirBnB de la ciudad de Oporto. Se puede encontrar más información sobre el dataset y otras implementaciones en [Porto](https://github.com/Vasallo94/Porto). 

El dataset contiene información sobre las características de las viviendas, su localización, el precio, el número de comentarios, etc.

El objetivo de este ejercicio es poder analizar los comentarios de los usuarios mediante LLMs y poder extraer información relevante de los mismos para su análisis posterior.

## Primera parte

Vamos a utilizar el archivo listings1_cleaned.csv que contiene información sobre las viviendas de AirBnB en Oporto.

In [11]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google import genai
from groq import Groq

# Carga variables desde .env del proyecto (si existe)
project_root = Path.cwd().parents[1]
load_dotenv(project_root / ".env")

LLM_PROVIDER = os.getenv("LLM_PROVIDER", "gemini").strip().lower()


def build_clients():
    gemini_client = None
    groq_client = None

    gemini_api_key = os.getenv("GEMINI_API_KEY")
    if gemini_api_key:
        gemini_client = genai.Client(api_key=gemini_api_key)

    groq_api_key = os.getenv("GROQ_API_KEY")
    if groq_api_key:
        groq_client = Groq(api_key=groq_api_key)

    return gemini_client, groq_client


gemini_client, groq_client = build_clients()

if LLM_PROVIDER == "gemini" and gemini_client is None:
    raise ValueError("LLM_PROVIDER=gemini pero no existe GEMINI_API_KEY en el entorno.")
if LLM_PROVIDER == "groq" and groq_client is None:
    raise ValueError("LLM_PROVIDER=groq pero no existe GROQ_API_KEY en el entorno.")


def load_raw_csv(filename: str) -> pd.DataFrame:
    p1 = Path(filename)
    p2 = project_root / "data" / "raw" / filename
    path = p1 if p1.exists() else p2
    return pd.read_csv(path)


def get_description_column(df: pd.DataFrame) -> str:
    if "description" in df.columns:
        return "description"

    lower_map = {c.lower().strip(): c for c in df.columns}
    for c in ["descripcion", "descriptions", "summary"]:
        if c in lower_map:
            return lower_map[c]

    raise KeyError(f"No se encontro columna de descripcion. Columnas: {list(df.columns)}")


def top_by_description_length(df: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    if not 10 <= top_n <= 20:
        raise ValueError("top_n debe estar entre 10 y 20")

    desc_col = get_description_column(df)
    out = df.copy()
    out[desc_col] = out[desc_col].fillna("").astype(str)
    out["description_length"] = out[desc_col].str.len()
    return out.sort_values("description_length", ascending=False).head(top_n)


listings = load_raw_csv("listings1_cleaned.csv")
listings_top = top_by_description_length(listings, top_n=10)
listings_top[[get_description_column(listings_top), "description_length"]].head()

,description,description_length
729,Fix breakfast in a cozy kitchen and dine at a ...,1000
883,Prep meals in the sleek kitchen with monochrom...,1000
835,Indulge in the luxury of classic Montreal arch...,1000
912,"Kick back on the crisp, white corner sofa at t...",1000
1035,You will enjoy sleeping in our soft and comfor...,1000


1. Filtra el dataset con una función que seleccione las 10/20 reseñas con mayor longitud (de string) en la columna descripción (las que consideraríamos las más 'relevantes')
2. Elabora un prompt de tal forma que un LLM sea capaz de extraer un json con las siguientes entidades de las entradas de la tabla filtrada:
```json
{
    "name": str,
    "location": str,
    "main_characteristics": str,
    "type": str,
    "size": str,
    "capacity": str,
    "key_amenities": list,
    "proximity_highlights": list,
}
```
3. Puedes elegir el proveedor que quieras, Gemini de Google o cualquiera de los modelos de Groq

## Parte 2

Realiza un análisis de los comentarios de los apartamentos con un LLM y extrae información relevante de los mismos.

Por ejemplo, puedes analizar los comentarios y extraer información sobre la limpieza, la ubicación, la relación calidad-precio, el sentimiento del comentario, etc. 

Igual que la parte 1 pero con un segundo dataset y con formato de salida a tu gusto

In [12]:
REQUIRED_KEYS = [
    "name",
    "location",
    "main_characteristics",
    "type",
    "size",
    "capacity",
    "key_amenities",
    "proximity_highlights",
]


def build_part1_prompt(description_text: str) -> str:
    return (
        "Extrae entidades de la descripcion de un alojamiento. "
        "Devuelve SOLO JSON valido sin markdown ni texto extra. "
        "Usa exactamente estas claves: "
        f"{', '.join(REQUIRED_KEYS)}. "
        "Si falta informacion, usa null o listas vacias.\n\n"
        f"Descripcion:\n{description_text}"
    )


def call_llm(prompt: str, json_mode: bool = False) -> str:
    if LLM_PROVIDER == "gemini":
        model = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")
        config = {"temperature": 0}
        if json_mode:
            config["response_mime_type"] = "application/json"

        response = gemini_client.models.generate_content(
            model=model,
            contents=prompt,
            config=config,
        )
        return (response.text or "").strip()

    model = os.getenv("GROQ_MODEL", "llama-3.1-70b-versatile")
    completion = groq_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": "Responde con JSON valido cuando se solicite JSON."},
            {"role": "user", "content": prompt},
        ],
    )
    return (completion.choices[0].message.content or "").strip()


def parse_json_or_none(text: str):
    if text.startswith("```"):
        text = text.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(text)
    except Exception:
        return None


def normalize_part1(payload: dict) -> dict:
    out = {k: payload.get(k) for k in REQUIRED_KEYS}

    for k in ["main_characteristics", "key_amenities", "proximity_highlights"]:
        if out[k] is None:
            out[k] = []
        elif not isinstance(out[k], list):
            out[k] = [str(out[k])]
        else:
            out[k] = [str(x) for x in out[k]]

    for k in ["name", "location", "type", "size", "capacity"]:
        if out[k] is not None:
            out[k] = str(out[k])

    return out


part1_results = []
desc_col = get_description_column(listings_top)

for idx, row in listings_top.iterrows():
    prompt = build_part1_prompt(str(row[desc_col]))
    raw = call_llm(prompt, json_mode=True)
    parsed = parse_json_or_none(raw)

    if isinstance(parsed, dict):
        item = normalize_part1(parsed)
    else:
        item = {k: None for k in REQUIRED_KEYS}
        item["main_characteristics"] = []
        item["key_amenities"] = []
        item["proximity_highlights"] = []
        item["raw_output"] = raw

    item["row_index"] = int(idx)
    item["description_length"] = int(row["description_length"])
    part1_results.append(item)

part1_df = pd.DataFrame(part1_results)
part1_df.head()

,name,location,main_characteristics,type,size,capacity,key_amenities,proximity_highlights,row_index,description_length
0,None,Little Italy,"[Sunny, 2 floors, 20' ceiling in the loft, Con...",Tiny-House,2 floors,NaN,"[Cozy kitchen, Scandi-style wood table, Skylig...",[Jean Talon market (less than a five minute wa...,729,1000
1,None,NaN,[Sleek kitchen with monochrome and marble acce...,home,NaN,5,"[Fully equipped kitchen, Stainless steel appli...","[Bakery, Supermarket]",883,1000
2,None,"NDG, Montreal","[Restored 1920s architecture (built in 1923), ...",Apartment,Very large,NaN,"[Private back patio deck, Weber propane barbec...",[],835,1000
3,None,NaN,"[brand-new, spacious, high-end, quiet, cozy, l...",condo,NaN,NaN,"[air conditioning, gray marble bathroom, crisp...",[],912,1000
4,None,Montreal,[Comfortable Queen-size bed with clean sheets ...,NaN,NaN,NaN,"[Queen-size bed, Sheets and pillows, Fully equ...","[Maxi grocery store (1 minute walk), Convenien...",1035,1000


In [13]:
# PARTE 2: Analisis de comentarios
reviews = load_raw_csv("Airbnb_reviews_5000.csv")

comment_col = "comments" if "comments" in reviews.columns else reviews.columns[0]
reviews[comment_col] = reviews[comment_col].fillna("").astype(str)
reviews["comment_length"] = reviews[comment_col].str.len()

# Tomamos comentarios largos para extraer señales mas ricas
reviews_top = reviews.sort_values("comment_length", ascending=False).head(20).copy()


def build_part2_prompt(comment: str) -> str:
    return (
        "Analiza este comentario de Airbnb y devuelve SOLO JSON valido con esta estructura exacta:\n"
        "{\n"
        '  "cleanliness": "positive|neutral|negative",\n'
        '  "location": "positive|neutral|negative",\n'
        '  "value_for_money": "positive|neutral|negative",\n'
        '  "host_interaction": "positive|neutral|negative",\n'
        '  "overall_sentiment": "positive|neutral|negative",\n'
        '  "key_points": ["string", "string"],\n'
        '  "summary": "string"\n'
        "}\n"
        "Sin texto extra, sin markdown.\n\n"
        f"Comentario:\n{comment}"
    )


part2_rows = []
for idx, row in reviews_top.iterrows():
    prompt = build_part2_prompt(str(row[comment_col]))
    raw = call_llm(prompt, json_mode=True)
    parsed = parse_json_or_none(raw)

    if not isinstance(parsed, dict):
        parsed = {
            "cleanliness": "neutral",
            "location": "neutral",
            "value_for_money": "neutral",
            "host_interaction": "neutral",
            "overall_sentiment": "neutral",
            "key_points": [],
            "summary": "No se pudo parsear salida del modelo.",
            "raw_output": raw,
        }

    parsed["row_index"] = int(idx)
    parsed["comment_length"] = int(row["comment_length"])
    part2_rows.append(parsed)

part2_df = pd.DataFrame(part2_rows)

print("Resumen sentimiento global (parte 2):")
print(part2_df["overall_sentiment"].value_counts(dropna=False))

part2_df.head()

Resumen sentimiento global (parte 2):
overall_sentiment
positive    12
negative     6
neutral      2
Name: count, dtype: int64


,cleanliness,location,value_for_money,host_interaction,overall_sentiment,key_points,summary,row_index,comment_length
0,negative,neutral,negative,negative,negative,"[El apartamento es bonito y tiene potencial, p...",Aunque el apartamento en sí es espacioso y bie...,2000,2976
1,positive,positive,positive,positive,positive,"[El anfitrión Fábio fue muy atento, respondió ...",Estancia muy positiva en un apartamento recien...,3876,2392
2,neutral,positive,neutral,positive,neutral,[Excelente ubicación cerca del metro y cafeter...,"El huésped valora positivamente la ubicación, ...",2456,2226
3,neutral,positive,neutral,positive,positive,"[Vistas impresionantes al río Duero, Oporto y ...",El huésped disfrutó mucho de su estancia desta...,1908,2011
4,negative,negative,negative,negative,negative,[Retraso en la limpieza y ausencia de llaves a...,Experiencia completamente negativa debido a re...,1636,1928


In [14]:
# Exportar resultados de ambas partes a data/processed (JSON estricto)
output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

part1_path = output_dir / "ej1_parte1_entidades.json"
part2_path = output_dir / "ej1_parte2_analisis.json"

if "part1_df" not in globals() or "part2_df" not in globals():
    raise RuntimeError("Ejecuta primero las celdas de Parte 1 y Parte 2 para generar part1_df y part2_df.")


def sanitize_missing(value):
    if isinstance(value, dict):
        return {k: sanitize_missing(v) for k, v in value.items()}
    if isinstance(value, list):
        return [sanitize_missing(v) for v in value]
    if pd.isna(value):
        return None
    return value


part1_records = part1_df.to_dict(orient="records")
part2_records = part2_df.to_dict(orient="records")

part1_records = sanitize_missing(part1_records)
part2_records = sanitize_missing(part2_records)

with open(part1_path, "w", encoding="utf-8") as f:
    json.dump(part1_records, f, ensure_ascii=False, indent=2, allow_nan=False)

with open(part2_path, "w", encoding="utf-8") as f:
    json.dump(part2_records, f, ensure_ascii=False, indent=2, allow_nan=False)

print(f"Archivo generado: {part1_path}")
print(f"Archivo generado: {part2_path}")

Archivo generado: c:\Users\venel\Documents\GitHub\IA0526\LLM\PRACTICA_OTROS_DATOS\data\processed\ej1_parte1_entidades.json
Archivo generado: c:\Users\venel\Documents\GitHub\IA0526\LLM\PRACTICA_OTROS_DATOS\data\processed\ej1_parte2_analisis.json
